## Map Type Columns in Spark Dataframes

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Map Type Columns in Spark Dataframes").getOrCreate()

In [2]:
import datetime
users_list = [
    {
        'id': 1,
        'first_name': 'John',
        'last_name': 'Doe',
        'email': 'john.doe@example.com',
        'phone_numer' : {'UAE_Mobile': '+971 54 7400113', 'PAK_Mobile': '+92 300 5638080'},
        'is_customer': True,
        'amount_paid': 123.45,
        'customer_from': datetime.date(2023, 1, 15),  # Date in YYYY-MM-DD format
        'last_updated_ts': datetime.datetime(2024, 1, 21, 20, 5, 46)  # Timestamp in YYYY-MM-DD HH:MM:SS format
    },
    {
        'id': 2,
        'first_name': 'Jane',
        'last_name': 'Smith',
        'email': 'jane.smith@example.com',
        'phone_numer' : {'UAE_Mobile': '+971 99 7666666', 'PAK_Mobile': '+92 300 0000000'},
        'is_customer': True,
        'amount_paid': 0.0,
        'customer_from': datetime.date(2022, 11, 10),
        'last_updated_ts': datetime.datetime(2024, 1, 21, 20, 5, 46)
    },
    {
        'id': 3,
        'first_name': 'Jack',
        'last_name': 'Alpha',
        'email': 'jack.alpha@example.com',
        'phone_numer' : {'UAE_Mobile': '+971 99 7666555'},
        'is_customer': False,
        'amount_paid': 0.0,
        'customer_from': datetime.date(2022, 11, 10),
        'last_updated_ts': datetime.datetime(2024, 1, 21, 20, 5, 46)
    }
    # Add more user dictionaries as needed
]

In [3]:
from pyspark.sql import Row

In [4]:
users_df = spark.createDataFrame([Row(**users) for users in users_list])

In [5]:
users_df

DataFrame[id: bigint, first_name: string, last_name: string, email: string, phone_numer: map<string,string>, is_customer: boolean, amount_paid: double, customer_from: date, last_updated_ts: timestamp]

In [6]:
users_df.printSchema()

root
 |-- id: long (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone_numer: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)
 |-- is_customer: boolean (nullable = true)
 |-- amount_paid: double (nullable = true)
 |-- customer_from: date (nullable = true)
 |-- last_updated_ts: timestamp (nullable = true)



In [7]:
users_df.dtypes

[('id', 'bigint'),
 ('first_name', 'string'),
 ('last_name', 'string'),
 ('email', 'string'),
 ('phone_numer', 'map<string,string>'),
 ('is_customer', 'boolean'),
 ('amount_paid', 'double'),
 ('customer_from', 'date'),
 ('last_updated_ts', 'timestamp')]

In [8]:
type(users_df)

pyspark.sql.classic.dataframe.DataFrame

In [9]:
users_df.show()

+---+----------+---------+--------------------+--------------------+-----------+-----------+-------------+-------------------+
| id|first_name|last_name|               email|         phone_numer|is_customer|amount_paid|customer_from|    last_updated_ts|
+---+----------+---------+--------------------+--------------------+-----------+-----------+-------------+-------------------+
|  1|      John|      Doe|john.doe@example.com|{PAK_Mobile -> +9...|       true|     123.45|   2023-01-15|2024-01-21 20:05:46|
|  2|      Jane|    Smith|jane.smith@exampl...|{PAK_Mobile -> +9...|       true|        0.0|   2022-11-10|2024-01-21 20:05:46|
|  3|      Jack|    Alpha|jack.alpha@exampl...|{UAE_Mobile -> +9...|      false|        0.0|   2022-11-10|2024-01-21 20:05:46|
+---+----------+---------+--------------------+--------------------+-----------+-----------+-------------+-------------------+



In [10]:
users_df.select('id', 'phone_numer').show(truncate=False)

+---+--------------------------------------------------------------+
|id |phone_numer                                                   |
+---+--------------------------------------------------------------+
|1  |{PAK_Mobile -> +92 300 5638080, UAE_Mobile -> +971 54 7400113}|
|2  |{PAK_Mobile -> +92 300 0000000, UAE_Mobile -> +971 99 7666666}|
|3  |{UAE_Mobile -> +971 99 7666555}                               |
+---+--------------------------------------------------------------+



In [11]:
users_df.columns

['id',
 'first_name',
 'last_name',
 'email',
 'phone_numer',
 'is_customer',
 'amount_paid',
 'customer_from',
 'last_updated_ts']

In [12]:
from pyspark.sql.functions import col

In [13]:
users_df.select('id', col('phone_numer')['UAE_Mobile'], col('phone_numer')['PAK_Mobile']) \
    .show()

+---+-----------------------+-----------------------+
| id|phone_numer[UAE_Mobile]|phone_numer[PAK_Mobile]|
+---+-----------------------+-----------------------+
|  1|        +971 54 7400113|        +92 300 5638080|
|  2|        +971 99 7666666|        +92 300 0000000|
|  3|        +971 99 7666555|                   NULL|
+---+-----------------------+-----------------------+



In [14]:
users_df. \
    select('id', col('phone_numer')['UAE_Mobile'].alias('International'), col('phone_numer')['PAK_Mobile'].alias('National')) \
    .show()

+---+---------------+---------------+
| id|  International|       National|
+---+---------------+---------------+
|  1|+971 54 7400113|+92 300 5638080|
|  2|+971 99 7666666|+92 300 0000000|
|  3|+971 99 7666555|           NULL|
+---+---------------+---------------+



In [15]:
from pyspark.sql.functions import explode

In [16]:
# alias cannot be given when using explode on the top of the map columns. It assigns the name as key and value

users_df \
    .select('id', explode('phone_numer')) \
    .show()

+---+----------+---------------+
| id|       key|          value|
+---+----------+---------------+
|  1|PAK_Mobile|+92 300 5638080|
|  1|UAE_Mobile|+971 54 7400113|
|  2|PAK_Mobile|+92 300 0000000|
|  2|UAE_Mobile|+971 99 7666666|
|  3|UAE_Mobile|+971 99 7666555|
+---+----------+---------------+



In [17]:
from pyspark.sql.functions import explode_outer

In [18]:
users_df \
    .select('id', explode_outer('phone_numer')) \
    .show()

+---+----------+---------------+
| id|       key|          value|
+---+----------+---------------+
|  1|PAK_Mobile|+92 300 5638080|
|  1|UAE_Mobile|+971 54 7400113|
|  2|PAK_Mobile|+92 300 0000000|
|  2|UAE_Mobile|+971 99 7666666|
|  3|UAE_Mobile|+971 99 7666555|
+---+----------+---------------+



In [19]:
users_df.select('*', explode('phone_numer')) \
    .withColumnRenamed('key', 'phone_type') \
    .withColumnRenamed('value', 'phone_number') \
    .drop('phone_numer') \
    .show(truncate=False)

+---+----------+---------+----------------------+-----------+-----------+-------------+-------------------+----------+---------------+
|id |first_name|last_name|email                 |is_customer|amount_paid|customer_from|last_updated_ts    |phone_type|phone_number   |
+---+----------+---------+----------------------+-----------+-----------+-------------+-------------------+----------+---------------+
|1  |John      |Doe      |john.doe@example.com  |true       |123.45     |2023-01-15   |2024-01-21 20:05:46|PAK_Mobile|+92 300 5638080|
|1  |John      |Doe      |john.doe@example.com  |true       |123.45     |2023-01-15   |2024-01-21 20:05:46|UAE_Mobile|+971 54 7400113|
|2  |Jane      |Smith    |jane.smith@example.com|true       |0.0        |2022-11-10   |2024-01-21 20:05:46|PAK_Mobile|+92 300 0000000|
|2  |Jane      |Smith    |jane.smith@example.com|true       |0.0        |2022-11-10   |2024-01-21 20:05:46|UAE_Mobile|+971 99 7666666|
|3  |Jack      |Alpha    |jack.alpha@example.com|false 

In [20]:
users_df.select('*', explode_outer('phone_numer')) \
    .withColumnRenamed('key', 'phone_type') \
    .withColumnRenamed('value', 'phone_number') \
    .drop('phone_numer') \
    .show(truncate=False)

+---+----------+---------+----------------------+-----------+-----------+-------------+-------------------+----------+---------------+
|id |first_name|last_name|email                 |is_customer|amount_paid|customer_from|last_updated_ts    |phone_type|phone_number   |
+---+----------+---------+----------------------+-----------+-----------+-------------+-------------------+----------+---------------+
|1  |John      |Doe      |john.doe@example.com  |true       |123.45     |2023-01-15   |2024-01-21 20:05:46|PAK_Mobile|+92 300 5638080|
|1  |John      |Doe      |john.doe@example.com  |true       |123.45     |2023-01-15   |2024-01-21 20:05:46|UAE_Mobile|+971 54 7400113|
|2  |Jane      |Smith    |jane.smith@example.com|true       |0.0        |2022-11-10   |2024-01-21 20:05:46|PAK_Mobile|+92 300 0000000|
|2  |Jane      |Smith    |jane.smith@example.com|true       |0.0        |2022-11-10   |2024-01-21 20:05:46|UAE_Mobile|+971 99 7666666|
|3  |Jack      |Alpha    |jack.alpha@example.com|false 